In [8]:
# Preliminaries -----------------------------------------------------------
if (!require("pacman")) install.packages("pacman")
pacman::p_load(tidyverse, ggplot2, dplyr, lubridate, haven)

library(knitr)
library(kableExtra)
     

In [12]:
hcris <- read_tsv("input/HCRIS_Data.txt")
kff <- read_csv("input/KFF_new_real.csv")

Rows: 161060 Columns: 44
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr   (9): provider_number, data_source, street, city, state, zip, county, n...
dbl  (31): beds, tot_charges, net_pat_rev, tot_discounts, tot_operating_exp,...
date  (4): fy_start, fy_end, date_processed, date_created

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 52 Columns: 4
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (4): Location, Status of Medicaid Expansion Decision, Expansion Implemen...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [14]:
kff.final <- kff %>%
  rename(
    state  = Location,
    status = `Status of Medicaid Expansion Decision`,
    expansion_date = `Expansion Implementation Date`
  ) %>%
  mutate(
    expansion_date = mdy(expansion_date)
  )

Warning message:
“There was 1 warning in `mutate()`.
ℹ In argument: `expansion_date = mdy(expansion_date)`.
Caused by warning:
!  10 failed to parse.”


In [17]:
state_xwalk <- tibble(
  state = state.name,
  abb   = state.abb
)

state_xwalk <- state_xwalk %>%
  add_row(state = "District of Columbia", abb = "DC")

kff.merge <- kff.final %>%
  left_join(state_xwalk, by = "state")

combined_hcris_mcaid <- hcris %>%
  left_join(kff.merge, by = c("state" = "abb")) %>% select(-state.y)

In [18]:
write_csv(combined_hcris_mcaid, "../data/output/combined_hcris_mcaid.csv")